In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
CATALOG = "workspace"
SCHEMA = "final"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_product"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_product"

In [0]:
product_df = spark.table(BRONZE_TABLE)

display(product_df)

In [0]:
product_df.printSchema()

In [0]:
print(f"Bronze Product Count : {product_df.count()}")

In [0]:
null_profile = product_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in product_df.columns
])

display(null_profile)

In [0]:
def clean_text(column_name):

    return trim(
        regexp_replace(
            regexp_replace(
                col(column_name),
                "\\+",
                ""
            ),
            "&",
            "and"
        )
    )

In [0]:
product_clean_df = (
    product_df
    .withColumn(
        "Name",
        initcap(clean_text("Name"))
    )
    .withColumn(
        "ProductNumber",
        upper(trim(col("ProductNumber")))
    )
)

In [0]:
product_clean_df = (
    product_clean_df
    .withColumn(
        "Color",
        when(
            col("Color").isNull(),
            "Unknown"
        ).otherwise(
            initcap(trim(col("Color")))
        )
    )
)

In [0]:
from pyspark.sql.functions import expr

product_clean_df = (
    product_clean_df

    .withColumn("ProductID",
                expr("try_cast(ProductID as BIGINT)"))

    .withColumn("MakeFlag",
                expr("try_cast(MakeFlag as INT)"))

    .withColumn("FinishedGoodsFlag",
                expr("try_cast(FinishedGoodsFlag as INT)"))

    .withColumn("SafetyStockLevel",
                expr("try_cast(SafetyStockLevel as INT)"))

    .withColumn("ReorderPoint",
                expr("try_cast(ReorderPoint as INT)"))

    .withColumn("StandardCost",
                expr("try_cast(StandardCost as DOUBLE)"))

    .withColumn("ListPrice",
                expr("try_cast(ListPrice as DOUBLE)"))

    .withColumn("Weight",
                expr("try_cast(Weight as DOUBLE)"))

    .withColumn("DaysToManufacture",
                expr("try_cast(DaysToManufacture as INT)"))

    .withColumn("ProductSubcategoryID",
                expr("try_cast(ProductSubcategoryID as INT)"))

    .withColumn("ProductModelID",
                expr("try_cast(ProductModelID as INT)"))
)

In [0]:
timestamp_columns = [
    "SellStartDate",
    "SellEndDate",
    "DiscontinuedDate",
    "ModifiedDate"
]

for c in timestamp_columns:
    product_clean_df = product_clean_df.withColumn(
        c,
        expr(f"try_cast({c} as timestamp)")
    )

In [0]:
bad_productid = product_clean_df.filter(
    col("ProductID").isNull()
).count()

print(f"Bad ProductID Records : {bad_productid}")

In [0]:
product_valid_df = product_clean_df.filter(
    col("ProductID").isNotNull()
)

product_valid_df = product_valid_df.filter(
    col("Name").isNotNull()
)

product_valid_df = product_valid_df.filter(
    col("ListPrice") >= 0
)

product_valid_df = product_valid_df.filter(
    col("StandardCost") >= 0
)

In [0]:
window_spec = (
    Window
    .partitionBy("ProductID")
    .orderBy(col("ingested_at").desc())
)

product_dedup_df = (
    product_valid_df
    .withColumn(
        "rn",
        row_number().over(window_spec)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
dq_productid = product_dedup_df.filter(
    col("ProductID").isNull()
).count()

print(
    f"ProductID Nulls : {dq_productid}"
)

dq_name = product_dedup_df.filter(
    col("Name").isNull()
).count()

print(
    f"Product Name Nulls : {dq_name}"
)

dq_price = product_dedup_df.filter(
    col("ListPrice") < 0
).count()

print(
    f"Negative Prices : {dq_price}"
)

dq_duplicate = (
    product_dedup_df
    .groupBy("ProductID")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    f"Duplicate Products : {dq_duplicate}"
)

In [0]:
product_dedup_df = (
    product_dedup_df
    .withColumn(
        "business_hash",
        sha2(
            concat_ws(
                "|",
                coalesce(col("Name"), lit("")),
                coalesce(col("Color"), lit("")),
                coalesce(col("ListPrice").cast("string"), lit("")),
                coalesce(col("StandardCost").cast("string"), lit(""))
            ),
            256
        )
    )
)

In [0]:
silver_product_df = (
    product_dedup_df
    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
    .withColumn(
        "silver_layer",
        lit("SILVER")
    )
)

In [0]:
(
    silver_product_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(SILVER_TABLE)
)

In [0]:
silver_df = spark.table(SILVER_TABLE)

print(
    f"Silver Product Count : {silver_df.count()}"
)

display(silver_df)